# portfolio_data — pull a table, fill in the blanks, hand it back

Everything in this package goes in and out as a pandas DataFrame.

```python
models = models_frame()                              # who is there
df     = blank_frame(models, sleeves=["equity"])     # a row per (model, sleeve, quarter)
df.loc[df.sleeve == "equity", "price_to_book"] = 2.87
upload_frame(df, source="Morningstar Direct")        # done
```

One row per (subject, sleeve, quarter). One column per thing you could measure, every one
of them blank until you fill it. **A blank cell means "not measured", which means "leave
whatever is already stored alone"** — so returns can land in a different pass from
characteristics without either wiping the other, and there is nothing to remember about
partial uploads.

Nothing here writes to the database until you set `DRY_RUN = False` in the next cell. Run
it top to bottom first; every write is guarded.

In [ ]:
from datetime import date

import pandas as pd

from portfolio_data import (
    # pull
    models_frame, holdings_frame, market_frame, stored_frame,
    # the template and the fill helper
    blank_frame, fill, template_columns,
    # push
    upload_frame, market_template, upload_market_frame, findings_frame,
    # odds and ends
    load_config, quarter_end_for_label, recent_quarter_ends, prune_orphans,
    KEY_COLUMNS, CONTEXT_COLUMNS, CHARACTERISTIC_COLUMNS, PERFORMANCE_COLUMNS,
    BREAKDOWN_COLUMNS,
)

DRY_RUN = True                                # <- flip to False when you mean it
AS_OF = quarter_end_for_label("Q1 2026")      # '2026-03-31'

cfg = load_config()
cfg.ensure_ready()          # raises with a fix-it message if SQLITE_DIR is wrong

print("database :", cfg.sqlite_dir)
print("uploads  :", cfg.portfolio_db.name)
print("quarters :", recent_quarter_ends(4))
print("as_of    :", AS_OF)

# Typical output
# --------------
# database : D:\Data\CRM
# uploads  : portfolio.sqlite
# quarters : ['2026-06-30', '2026-03-31', '2025-12-31', '2025-09-30']
# as_of    : 2026-03-31
#
# There is nothing to configure in a checkout. SQLITE_DIR is read from the environment or
# from the same .env the Next.js app uses, resolved nearest-first: ./.env, backend/.env,
# then the repo root.

## 1. Pulling

Three things come out of the CRM:

| | |
|---|---|
| `models_frame()` | one row per logged client model — who, how big, how many positions |
| `holdings_frame()` | one row per position, per sleeve |
| `market_frame()` | Treasury par yields and credit spreads previously uploaded |

Every model arrives split into three portfolios: the **total**, the **equity** sleeve
rescaled to 100%, and the **fixed income** sleeve rescaled to 100%. That split is the whole
point — an equity style run over a book that is 38% bonds produces a price-to-book that
describes nothing, and a duration over the same book is diluted by every share in it.

Only those three can be pulled. The `equity_us` / `equity_developed` / `equity_em` sleeves
are **upload-only**: a holding record carries an identifier, an asset class and a weight,
and no domicile. Splitting the equity book by region needs a security master, which is your
end of the job — so you compute those slices and upload them under those names.

In [ ]:
models = models_frame()

print(models.shape)
models.head(3)

# Typical parameters — all eight filters, which AND together
# ---------------------------------------------------------
# models_frame(
#     crn="MOCK-000001",             # one external client
#     model_ids=[...],               # re-pull a specific batch
#     departments=["Brokerage"],     # the client department on the logging interaction
#     offices=["Office A"],
#     teams=["Default Team"],
#     min_aum=1_000_000_000,         # strict lower bound, in dollars
#     main_only=True,                # each client's main model — the "Avg. Client" cohort
#     logged_since="2026-01-01",     # last logged on or after this date
# )
#
# Careful with min_aum: a model whose AUM was never entered satisfies *no* threshold,
# because SQL leaves NULL out of every comparison. Those exclusions are logged rather than
# left to read as "nothing matched".
#
# Typical output
# --------------
# (69, 16)
#                              subject_id          crn        client_name          model_name  is_main       aum    client_dept  ...
# 0  14c0a00a-54cf-47d4-b96b-45fb47e18648  MOCK-000001  Vanguard Advisors          Core Model     True  10000000  Institutional
# 1  e97ab79c-2a8c-4a99-89be-9a64a668d0d4  MOCK-000001  Vanguard Advisors        Growth Model    False  25000000      Brokerage
# 2  ab2b3418-fd07-47a9-b35f-ea40111b5b6b  MOCK-000001  Vanguard Advisors  Conservative 60/40    False      <NA>       Advisory
#
# `subject_id` is named for what it becomes: hand this frame to blank_frame() and the ids
# carry through to the upload with no translation. <NA> is a model with no AUM recorded.

In [ ]:
equity = holdings_frame(models, sleeves=["equity"])

tickers = equity.identifier.unique()          # what you hand a security master
print(len(equity), "positions,", len(tickers), "distinct identifiers")
equity.head(4)

# Typical output
# --------------
#                              subject_id    model_name  sleeve  weight_of_total identifier constituent_type asset_class    weight  portfolio_weight
# 0  14c0a00a-54cf-47d4-b96b-45fb47e18648    Core Model  equity         0.778885       FMIC         Security      Equity  0.337336          0.262746
# 1  14c0a00a-54cf-47d4-b96b-45fb47e18648    Core Model  equity         0.778885       FMVX         Security      Equity  0.414537          0.322877
# 2  14c0a00a-54cf-47d4-b96b-45fb47e18648    Core Model  equity         0.778885       FMAS         Security      Equity  0.248128          0.193263
# 3  e97ab79c-2a8c-4a99-89be-9a64a668d0d4  Growth Model  equity         0.850000        VEA         Security      Equity  0.250000          0.212500
#
# Three weights, three meanings:
#   weight            within the sleeve — always sums to 1.0 per (model, sleeve)
#   weight_of_total   the sleeve's share of the whole portfolio
#   portfolio_weight  the two multiplied — the position's real weight in the client's book

In [ ]:
holdings = holdings_frame(models)             # all three sleeves

print(holdings.groupby(["subject_id", "sleeve"]).weight.sum().head(6))

shares = models[["equity_weight_of_total", "fixed_income_weight_of_total"]].sum(axis=1)
print("\nsleeve shares that fall short of 1.0:", int((shares < 0.999).sum()), "of", len(models))

# Typical output
# --------------
# subject_id                            sleeve
# 14c0a00a-54cf-47d4-b96b-45fb47e18648  total           1.0
#                                       equity          1.0
#                                       fixed_income    1.0
#
# Each sleeve sums to 1.0 because each is rescaled to stand on its own.
#
# The equity and fixed income shares do NOT have to add to 1.0, and usually don't. A
# balanced fund or a multi-asset holding cannot be decomposed into stocks and bonds from a
# holding record alone, so it stays in `total` and belongs to neither sleeve. That residual
# is real information, not a rounding error — do not normalise it away.

In [ ]:
curve = market_frame(series=["ust_par_yield"])
print(curve.tail(3).to_string(index=False))

have = set(market_frame(series=["ig_oas"]).as_of)     # the usual reason to read these:
print("\nig_oas dates already stored:", len(have))    # find the gap before a backfill

# Typical output
# --------------
#        series tenor      as_of   value    source
# ust_par_yield   10Y 2025-09-30 0.04002 DEMO SEED
# ust_par_yield    1M 2025-09-30 0.04312 DEMO SEED
# ust_par_yield    1Y 2025-09-30 0.03744 DEMO SEED
#
# These are market-level, not per-model: a yield curve belongs to the market, not to
# anybody's portfolio. Nothing populates them but you — see the last section.

In [ ]:
stored = stored_frame(as_of=AS_OF)

print(stored.shape)
print("\ncoverage — how many rows carry each metric:")
print(stored[list(CHARACTERISTIC_COLUMNS)].notna().sum().sort_values(ascending=False).head(8))

# Typical output
# --------------
# (410, 82)
#
# num_holdings          405
# expense_ratio         345
# median_market_cap     280
# wtd_avg_market_cap    280
# price_to_book         280
# ...
#
# stored_frame() is what has already been uploaded, in exactly the template's shape (plus a
# trailing `uploaded_at`). That identity is deliberate:
#
#     upload_frame(stored_frame(as_of=AS_OF))
#
# is a no-op round trip, which makes read-modify-write the natural way to correct three
# cells of last quarter without rebuilding anything.

## 2. The template

`blank_frame()` gives you one row per (subject, sleeve, quarter) and one column per thing
you could measure. Eighty-one columns, in seven families:

| Family | Columns | On upload |
|---|---|---|
| key | `subject_kind`, `subject_id`, `sleeve`, `as_of` | required |
| context | `crn`, `client_name`, `model_name`, `aum`, `quarter`, … | ignored — they are there so you can read the table |
| characteristics | `price_to_book`, `effective_duration`, `underlying_companies`, … (18) | written |
| performance | `return_1y`, `sharpe_3y`, `benchmark_id`, … (18) | written |
| breakdown buckets | `region.US`, `credit_rating.AAA`, … (29) | written |
| holding counts | `names.region.US`, … (29) | written — opt in with `include_names=True` |
| `source` | where the numbers came from | written |
| `_anything` | yours | ignored |

**Blank means "leave it alone."** An empty cell is written as SQL NULL and the upsert
coalesces it away, so it never overwrites a stored value. Fill what you have; ignore the
rest.

**Breakdowns are the one exception.** Buckets are replaced wholesale per dimension, so:

- leave a dimension **entirely** blank and it is not sent at all — whatever is stored stays;
- fill it **completely** and it replaces what was there;
- fill it **halfway** and it is rejected, because a distribution summing to 0.6 draws a
  chart that quietly stops short of the edge.

**A misspelled column is an error, not a shrug.** `price_to_books` raises before anything
opens, naming the column you probably meant. The alternative — dropping it — is an upload
that reports success and stores nothing.

In [ ]:
tpl = blank_frame(models.head(2), sleeves=["equity", "fixed_income"], as_of=AS_OF)

print(tpl.shape)
tpl[["subject_kind", "subject_id", "sleeve", "as_of", "client_name",
     "model_name", "quarter", "sleeve_positions", "price_to_book"]]

# Typical parameters
# ------------------
# blank_frame(
#     models,                            # a models_frame, a list of LoggedModel, or None
#     sleeves=["total", "equity", "fixed_income"],   # add equity_us / equity_developed / equity_em
#     as_of="Q1 2026",                   # or '2026-03-31', or a list of either
#     include_benchmarks=False,          # add a row per index as well
#     include_names=False,               # add the 29 holding-count columns
#     only_populated_sleeves=True,       # skip a sleeve a model has no holdings in
# )
#
# Typical output
# --------------
# (4, 81)
#   subject_kind                            subject_id        sleeve       as_of        client_name    model_name  quarter  sleeve_positions  price_to_book
# 0        model  14c0a00a-54cf-47d4-b96b-45fb47e18648        equity  2026-03-31  Vanguard Advisors    Core Model  Q1 2026                 3            NaN
# 1        model  14c0a00a-54cf-47d4-b96b-45fb47e18648  fixed_income  2026-03-31  Vanguard Advisors    Core Model  Q1 2026                 1            NaN
# 2        model  e97ab79c-2a8c-4a99-89be-9a64a668d0d4        equity  2026-03-31  Vanguard Advisors  Growth Model  Q1 2026                 4            NaN
# 3        model  e97ab79c-2a8c-4a99-89be-9a64a668d0d4  fixed_income  2026-03-31  Vanguard Advisors  Growth Model  Q1 2026                 1            NaN
#
# 81 columns render badly across a notebook, so select the ones you care about — or
# transpose a couple of rows with tpl.head(2).T to read a whole record top to bottom.

In [ ]:
for label, group in [("key", KEY_COLUMNS), ("context", CONTEXT_COLUMNS),
                     ("characteristics", CHARACTERISTIC_COLUMNS),
                     ("performance", PERFORMANCE_COLUMNS),
                     ("breakdown buckets", BREAKDOWN_COLUMNS)]:
    print(f"{label} ({len(group)}):")
    print("   ", ", ".join(group), "\n")

# Typical output — this is the complete list of what you can fill in
# ------------------------------------------------------------------
# key (4):
#     subject_kind, subject_id, sleeve, as_of
#
# context (11):
#     crn, client_name, model_name, is_main, aum, client_dept, logged_team, logged_office,
#     quarter, sleeve_weight_of_total, sleeve_positions
#
# characteristics (18):
#     wtd_avg_market_cap, median_market_cap, price_to_book, price_to_earnings,
#     price_to_sales, profitability, dividend_yield, return_on_equity,
#     underlying_companies, effective_duration, effective_maturity, yield_to_maturity,
#     sec_yield, avg_coupon, avg_credit_quality, num_holdings, expense_ratio, turnover
#
# performance (18):
#     return_qtd, return_ytd, return_1y, return_3y, return_5y, return_10y,
#     return_since_inception, std_dev_3y, sharpe_3y, beta_3y, alpha_3y, r_squared_3y,
#     tracking_error_3y, information_ratio_3y, up_capture_3y, down_capture_3y,
#     max_drawdown, benchmark_id
#
# breakdown buckets (29):
#     region.US, region.Developed ex-US, region.Emerging Markets,
#     market_cap.Large, market_cap.Mid, market_cap.Small,
#     style.Value, style.Blend, style.Growth,
#     credit_rating.AAA, credit_rating.AA, credit_rating.A, credit_rating.BBB,
#     credit_rating.BB, credit_rating.B, credit_rating.CCC & Below, credit_rating.Not Rated,
#     security_type.Government, security_type.Municipal, security_type.Corporate,
#     security_type.Securitized, security_type.Cash & Equivalents,
#     maturity_band.0-1Y, maturity_band.1-3Y, maturity_band.3-5Y, maturity_band.5-7Y,
#     maturity_band.7-10Y, maturity_band.10-20Y, maturity_band.20Y+
#
# Bucket names are verbatim, spaces and '&' included, so use df["region.Developed ex-US"]
# rather than attribute access. Two counts that are easy to confuse:
#   num_holdings          line items in the portfolio
#   underlying_companies  distinct issuers, looking through the funds

## 3. Filling it in

Three ways, in rough order of how often you will want them.

**Units, every time:** everything that is conceptually a percentage is a decimal fraction.
`8.4%` is `0.084`. A `return_1y` of `8.4` is rejected, because 840% and 8.4% look equally
plausible once they are sitting in a database. The exceptions are the genuinely unitless
statistics — beta, Sharpe, R-squared, information ratio — and market caps, which are
dollars.

In [ ]:
# (a) By mask — one value across every row that matches.
tpl.loc[tpl.sleeve == "equity", "price_to_book"] = 2.87
tpl.loc[tpl.sleeve == "equity", "underlying_companies"] = 3184
tpl.loc[tpl.sleeve == "fixed_income", "effective_duration"] = 6.1        # years
tpl.loc[tpl.sleeve == "fixed_income", "yield_to_maturity"] = 0.0455      # 4.55%

# (b) A whole breakdown at once — all of a dimension's buckets in one assignment.
tpl.loc[tpl.sleeve == "equity",
        ["region.US", "region.Developed ex-US", "region.Emerging Markets"]] = [0.62, 0.28, 0.10]

# One model's numbers, by key.
mid = tpl.subject_id.iloc[0]
tpl.loc[(tpl.subject_id == mid) & (tpl.sleeve == "equity"), "return_1y"] = 0.084

tpl[["subject_id", "sleeve", "price_to_book", "underlying_companies",
     "effective_duration", "region.US", "return_1y"]]

In [ ]:
# (c) From your engine's own frame. It needs the key columns plus whatever it computed;
#     fill() matches on the key and copies every non-blank cell across.
results = pd.DataFrame({
    "subject_id": [mid],
    "sleeve": ["equity"],
    "as_of": [AS_OF],
    "price_to_earnings": [21.4],
    "num_holdings": [412],
})

tpl = fill(tpl, results)
tpl[["subject_id", "sleeve", "price_to_book", "price_to_earnings", "num_holdings"]]

# Typical parameters
# ------------------
# fill(template, values, add_missing=False)
#
# Blank cells in `values` are left alone, so two engines can fill the same template in
# either order. Returns a new frame; neither argument is modified.
#
# A key in `values` that matches no template row RAISES:
#
#     ValueError: 1 row(s) in `values` match no template row, e.g.
#     ('model', 'not-a-model', 'equity', '2026-03-31'). Pass add_missing=True to append
#     them, or check the keys — an unmatched key would otherwise be silently dropped.
#
# which is the entire reason this exists rather than DataFrame.update: update is
# index-aligned and drops unmatched rows without a word.

In [ ]:
# Two things worth eyeballing before you upload.
print("cells filled per column:")
print(tpl.notna().sum()[lambda s: s > 0].drop(list(KEY_COLUMNS) + list(CONTEXT_COLUMNS)))

print("\nbreakdown sums (must be 1.0, or 0.0 for a dimension you are leaving alone):")
for dimension in ("region", "style", "credit_rating"):
    print(f"  {dimension:<14}", tpl.filter(like=dimension + ".").sum(axis=1).tolist())

# Typical output
# --------------
# cells filled per column:
# price_to_book             2
# underlying_companies      2
# effective_duration        2
# region.US                 2
# ...
#
# breakdown sums (must be 1.0, or 0.0 for a dimension you are leaving alone):
#   region         [1.0, 0.0, 1.0, 0.0]
#   style          [0.0, 0.0, 0.0, 0.0]
#   credit_rating  [0.0, 0.0, 0.0, 0.0]
#
# A value between the two — 0.62, say — is a half-filled dimension, and will be rejected.

## 4. Uploading

Dry-run first, always. It runs the full validation matrix against live data and writes
nothing, which is exactly what you want the first time a new export is wired up.

Two kinds of problem, handled two ways:

- **Something wrong with the table** — an unknown column, a blank or duplicated key, a
  holding count with no weight beside it — raises immediately and writes nothing at all.
- **Something wrong with a value** — a percent where a fraction belongs, a breakdown that
  does not sum, a model that has since been deleted — is recorded against that row and the
  rest of the batch continues.

Exit codes, for a scheduler: `0` clean · `1` some rows failed · `2` could not start ·
`3` everything failed.

In [ ]:
summary = upload_frame(
    tpl,
    source=f"Notebook {date.today()}",     # where these numbers came from
    dry_run=True,
)

print(summary.render())
findings_frame(summary)

# Typical output
# --------------
# ==================================================================
# portfolio_data upload summary  [DRY RUN - nothing written]
# ==================================================================
#   records seen : 4
#   validated    : 4
#   skipped      : 0  (nothing to write)
#   failed       : 0
#   exit code    : 0
# ==================================================================
#
# `skipped` counts rows you left entirely blank — expected, since a template is mostly
# blank by construction. They are dropped before the upload so they cannot bury the
# findings that matter.
#
# A failure looks like this, and names the row:
#   failures:
#     - model:14c0a00a... / equity @ 2026-03-31: [ERROR] return_1y: performance.return_1y
#       is 8.4, which reads as 840%. These are stored as decimal fractions.

In [ ]:
if not DRY_RUN:
    summary = upload_frame(tpl, source=f"Notebook {date.today()}")
    print(summary.render())
    print("exit code:", summary.exit_code)
else:
    print("DRY_RUN is True — nothing written.")
    print("Set DRY_RUN = False in the setup cell at the top, then re-run this cell.")

In [ ]:
# Read back exactly what you just wrote.
check = stored_frame(as_of=AS_OF, subject_ids=list(tpl.subject_id.unique()))
check[["subject_id", "sleeve", "price_to_book", "underlying_companies",
       "region.US", "source", "uploaded_at"]]

# Nothing there? Two usual reasons: DRY_RUN is still True, or the as_of you filled in is
# not the one you are reading back.

## 5. Benchmarks

Every card on the Portfolio Trends page is captioned "vs \<index\>", so an index's numbers
live in the same tables as the models and are queried the same way. They travel the same
path with `subject_kind="benchmark"`.

Five are registered out of the box:

| sleeve | benchmark |
|---|---|
| `total`, `equity` | `MSCI-ACWI-IMI` |
| `equity_us` | `RUSSELL-3000` |
| `equity_developed` | `MSCI-WORLD-EX-USA-IMI` |
| `equity_em` | `MSCI-EM-IMI` |
| `fixed_income` | `BBG-US-AGG` |

They are off by default because the numbers are identical for every client and usually
somebody else's job — leaving them in would mean deleting the same rows every time.

In [ ]:
bench = blank_frame(models.head(1), sleeves=["equity", "fixed_income"],
                    as_of=AS_OF, include_benchmarks=True)

bench[bench.subject_kind == "benchmark"][["subject_kind", "subject_id", "model_name", "sleeve", "as_of"]]

# Typical output
# --------------
#   subject_kind     subject_id              model_name        sleeve       as_of
# 0    benchmark     BBG-US-AGG  Bloomberg US Aggregate  fixed_income  2026-03-31
# 1    benchmark  MSCI-ACWI-IMI           MSCI ACWI IMI        equity  2026-03-31
#
# Fill and upload them exactly like model rows. An unregistered benchmark id is rejected
# rather than stored, so a typo cannot become a subject nothing compares against.

## 6. Regional equity sleeves

`equity_us`, `equity_developed` and `equity_em` are upload-only. `get_models()` cannot
produce them — a holding record carries no domicile — so you split the equity book yourself
and upload each slice under its own name.

They behave like `equity` everywhere else: same metrics, same dimensions, same validation.
Only the benchmark differs, because a US-only book measured against an all-country index
would report a US overweight that is an artifact of the scope rather than a decision anyone
made. A `region` breakdown on one of them is redundant by construction — the region *is*
the sleeve.

In [ ]:
regional = blank_frame(models.head(1),
                       sleeves=["equity_us", "equity_developed", "equity_em"],
                       as_of=AS_OF,
                       only_populated_sleeves=False)     # no sleeve to check against

regional.loc[regional.sleeve == "equity_us", "price_to_book"] = 3.4
regional.loc[regional.sleeve == "equity_us",
             ["style.Value", "style.Blend", "style.Growth"]] = [0.31, 0.36, 0.33]

print(upload_frame(regional, source="Notebook regional split", dry_run=True).render())

# only_populated_sleeves=False matters here: there is no equity_us sleeve on a LoggedModel
# to check for holdings, so the default would drop every row.

## 7. Market series

Treasury par yields and credit spreads. Not per-model and not per-sleeve — a yield curve
belongs to the market, not to anybody's portfolio — so they get their own pair of functions.

Three things differ from model data:

- **`as_of` is not restricted to quarter ends.** The curve is naturally daily and the
  credit-spread card plots a history.
- **Units differ per series.** Yields are decimal fractions (`0.0435` for 4.35%); OAS
  spreads are basis points (`94`, not `0.0094`).
- **A series with no term structure uses `tenor=''`**, never NULL — the tenor is part of the
  primary key, and SQLite would let a NULL one silently duplicate the same day.

In [ ]:
curve = market_template(series=["ust_par_yield"], as_of="2026-03-31")

curve.loc[curve.tenor == "2Y", "value"] = 0.0412       # 4.12%, as a fraction
curve.loc[curve.tenor == "10Y", "value"] = 0.0435

spreads = market_template(series=["ig_oas", "hy_oas"], as_of="2026-03-31")
spreads["value"] = [94.0, 312.0]                       # basis points

print(upload_market_frame(pd.concat([curve, spreads], ignore_index=True),
                          source="Notebook market pull", dry_run=True).render())

# Typical output
# --------------
#   records seen : 13
#   validated    : 4
#   skipped      : 9  (nothing to write)
#   failed       : 0
#
# market_template() produces a row per valid tenor — eleven for ust_par_yield, one each for
# the spreads — so the nine you left blank are skipped rather than written as nulls.
#
# if not DRY_RUN:
#     upload_market_frame(curve, source="Notebook market pull")

## 8. Housekeeping

In [ ]:
# Analytics whose model has since been deleted. subject_id points at a table in a different
# SQLite file, so there is no foreign key to catch it, and orphans inflate every rollup
# that counts rows. Defaults to a dry run.
print(prune_orphans())
# prune_orphans(dry_run=False)         # actually delete

# Other things worth knowing about
# --------------------------------
#   python -m portfolio_data.test              full smoke test; cleans up after itself
#   python backend/seed_portfolio_analytics.py populate demo analytics for a few quarters
#
# Going through Excel or CSV? Pin the text columns on the way back in, or a numeric-looking
# subject_id becomes a float and an as_of becomes a Timestamp:
#
#   pd.read_csv("filled.csv", dtype={"subject_id": "string", "as_of": "string",
#                                    "sleeve": "string", "subject_kind": "string"})
#
# upload_frame() copes with a Timestamp as_of, but it cannot restore an id that Excel
# already mangled.

## Advanced: the dataclass API

Everything above is a thin layer over these, which remain fully supported. They are what
you want inside a scheduled job, where a frame buys you nothing:

```python
from portfolio_data import (PortfolioData, Characteristics, Performance, Breakdown,
                            get_models, upload_pf_data)

for model in get_models():
    upload_pf_data(PortfolioData(
        subject_id=model.id, sleeve="equity", as_of=AS_OF,
        characteristics=Characteristics(price_to_book=2.87, underlying_companies=3184),
        performance=Performance(return_1y=0.084, benchmark_id="MSCI-ACWI-IMI"),
        breakdowns=[Breakdown("region", {"US": 0.62, "Developed ex-US": 0.28,
                                         "Emerging Markets": 0.10})],
        source="Morningstar Direct 2026-04-02",
    ))
```

`upload_frame()` builds exactly these objects and calls exactly this function, so the
validation, the verify-after-write and the per-record failure isolation are the same code
either way.

Full account, including the validation matrix and what each rule exists to prevent:
`backend/portfolio_data/docs/README.md`.